# FXS / 06 — The positive structures on the sealed 2026 data: one look

Protocol: [`004_claude_fxs06_portfolio_holdout_preregistration.md`](../../../discussions/FX_Intraday_Seasonality/004_claude_fxs06_portfolio_holdout_preregistration.md).

| Structure | Rule (every trading day, news days included) | Promised net pips | Promised USD / lot |
|---|---|---|---|
| EURUSD-02 | sell 04:00 → buy 08:00 New York | +1.53 | +15.30 |
| USDJPY-01 | sell 09:55 → buy 10:55 JST | +0.92 | +6.36 |
| USDJPY-02 | buy 09:25 → sell 09:55 JST | +0.74 | +5.72 |
| EURJPY-02 | buy 12:35 → sell 16:35 New York | +0.83 | +5.91 |

Per structure and for the 1-lot-each portfolio: `HOLDS` (mean > 0, not significantly below promised) · `INCONCLUSIVE` (mean ≤ 0, not significantly below) · `FAILS` (significantly below promised). USD/JPY's 2026 Tokyo mornings were partly seen in GTF 02 (disclosed).

**Run once; commit `Portfolio/05/` without changing anything.**

In [ ]:
import json, os, sys, tempfile
import datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

START, END = dt.date(2026, 1, 1), dt.date(2026, 11, 6)
COMMISSION_PER_LOT = 3.0
E0 = 6_000.0
PROMISED_PIPS = {"EURUSD-02": 1.53, "USDJPY-01": 0.92, "USDJPY-02": 0.74, "EURJPY-02": 0.83}
PROMISED_USD = {"EURUSD-02": 15.30, "USDJPY-01": 6.36, "USDJPY-02": 5.72, "EURJPY-02": 5.91}
IDS = list(PROMISED_PIPS)

SYNTHETIC = os.environ.get("FXS_SYNTHETIC") == "1"
if SYNTHETIC:
    os.environ["EQUITY_TRADING_DIR"] = os.environ.get("FXS_SYNTH_DIR") or tempfile.mkdtemp(prefix="fxs06_")
    os.environ.pop("DUKASCOPY_DIR", None)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "equity_trading").exists())
sys.path.insert(0, str(ROOT / "src"))
from equity_trading import fx_seasonality as F
from scipy.stats import norm

cfg = F.FXSConfig()
BRANCH = Path.cwd().parent
OUT = Path.cwd() / ("05_synthetic" if SYNTHETIC else "05"); OUT.mkdir(exist_ok=True)
FROZEN = {"EURUSD-02": ("NY", 660, 900, -1), "USDJPY-01": ("UTC", 55, 115, -1),
          "USDJPY-02": ("UTC", 25, 55, 1), "EURJPY-02": ("NY", 1175, 1415, 1)}

## 0. Synthetic inputs (smoke run only)

In [ ]:
if SYNTHETIC:
    base = Path(os.environ["EQUITY_TRADING_DIR"]) / "Dukascopy"
    for i, p in enumerate(("EURUSD", "USDJPY", "EURJPY")):
        F.write_synthetic_candles(base, p, dt.date(2025, 12, 28), END, seed=2026 + i, drift_bps=-3.0 if p == "EURUSD" else 0.0,
                                  drift_ny=(4, 8), price=150.0 if p.endswith("JPY") else 1.10)
    cand = pd.DataFrame([{"id": k, "pair": k[:6], "clock": c, "start": s, "end": e, "direction": d} for k, (c, s, e, d) in FROZEN.items()])
else:
    cand = F.load_frozen_candidates(BRANCH).query("id in @IDS").reset_index(drop=True)
for _, r in cand.iterrows():
    assert (r["clock"], int(r["start"]), int(r["end"]), int(r["direction"])) == FROZEN[r["id"]], f"{r['id']} is not the frozen rule"
cand["rule"] = [f"{'buy' if d > 0 else 'sell'} {F.slot_label(c, s)} -> {F.slot_label(c, e)}" for c, s, e, d in
                zip(cand["clock"], cand["start"], cand["end"], cand["direction"])]
display(cand[["id", "pair", "rule"]])

## 1. Data complete?

In [ ]:
PAIRS = sorted(set(cand["pair"]))
for p in PAIRS:
    cov = F.file_coverage(p, START, END)
    print(f"{p} 2026: {cov['present']}/{cov['expected']} day files ({cov['share']:.2%}); missing {[str(d) for d in cov['missing']]}")
    assert cov["share"] >= cfg.min_file_share, f"{p}: download not complete; finish it before the one look"
panels = F.load_panels(PAIRS, START, END, "holdout", cfg)

## 2. The one look: each structure

In [ ]:
G, M, P = F.structure_detail(cand, START, END, "holdout", cfg, panels=panels, with_price=True)
PAIR_OF = cand.set_index("id")["pair"]
traded = G != 0

def verdict_of(x, promised):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    m, se, t = (float(v[0]) for v in F.nw_tstats(x[:, None], cfg.nw_lags))
    p_pos, p_below = float(norm.sf(m / se)), float(norm.cdf((m - promised) / se))
    h1, h2, h3 = m > 0, p_below <= 0.05, p_pos <= 0.05
    return {"n": len(x), "mean": m, "se": se, "t": m / se, "p_mean_gt_0": p_pos, "p_below_promised": p_below,
            "H1": bool(h1), "H2": bool(h2), "H3": bool(h3), "verdict": "FAILS" if h2 else ("HOLDS" if h1 else "INCONCLUSIVE")}

net_pips, net_usd, rows = {}, {}, []
for i in IDS:
    pair, comm = PAIR_OF[i], F.commission_bps_per_lot(PAIR_OF[i], COMMISSION_PER_LOT)
    d = traded[i]
    gp = F.bps_to_pips(G[i][d], P[i][d], pair); cp = F.bps_to_pips(comm, P[i][d], pair)
    net_pips[i] = pd.Series(gp - cp, index=G.index[d])
    net_usd[i] = pd.Series((gp - cp) * F.usd_per_pip_per_lot(pair, P[i][d]), index=G.index[d])
    v = verdict_of(net_pips[i], PROMISED_PIPS[i])
    rows.append({"id": i, "promised_pips": PROMISED_PIPS[i], "pips_after_spread": float(np.mean(gp)), "commission_pips": float(np.mean(cp)),
                 "net_pips": v["mean"], "usd_per_lot_trade": float(net_usd[i].mean()), "win_rate": float((net_pips[i] > 0).mean()),
                 "typical_swing_pips": float(net_pips[i].std()), "worst_open_loss_pips": float(F.bps_to_pips(M[i][d], P[i][d], pair).min()),
                 **{k: v[k] for k in ("n", "t", "p_mean_gt_0", "p_below_promised", "H1", "H2", "H3", "verdict")}})
per = pd.DataFrame(rows).set_index("id")
display(per.round(3)); per.to_csv(OUT / "05_per_structure.csv")

## 3. The one look: the portfolio (1 lot of each structure, USD per day)

In [ ]:
daily = pd.DataFrame(net_usd).fillna(0.0)
port = daily.sum(axis=1)
pv = verdict_of(port, sum(PROMISED_USD.values()))
pv.update({"promised_usd_per_day": sum(PROMISED_USD.values()), "total_usd": float(port.sum()), "win_rate_days": float((port > 0).mean()),
           "worst_day_usd": float(port.min()), "best_day_usd": float(port.max())})
display(pd.Series(pv).to_frame("portfolio, 1 lot each"))
display(daily.corr().round(2))
fig, ax = plt.subplots(figsize=(11, 4))
for i in IDS:
    ax.plot(daily.index, daily[i].cumsum(), lw=0.9, label=i)
ax.plot(daily.index, port.cumsum(), color="black", lw=1.6, label="portfolio")
ax.plot(daily.index, sum(PROMISED_USD.values()) * np.arange(1, len(port) + 1), "k--", lw=0.7, label="promised")
ax.axhline(0, color="black", lw=0.5); ax.set_ylabel("USD, 1 lot each"); ax.legend(ncol=3)
plt.tight_layout(); plt.savefig(OUT / "05_cumulative_usd.png", dpi=110); plt.show()

## 4. By month, and 2024–2026 side by side (net pips per trade)

In [ ]:
monthly = pd.DataFrame({i: net_pips[i].groupby(net_pips[i].index.to_period("M")).mean() for i in IDS})
monthly["portfolio USD/day"] = port.groupby(port.index.to_period("M")).mean()
display(monthly.round(2)); monthly.to_csv(OUT / "05_monthly.csv")
if not SYNTHETIC:
    Gp, Mp, Pp = F.structure_detail(cand, *F.CONFIRMATION, "confirmation", cfg, with_price=True)
    yr = {}
    for i in IDS:
        d = Gp[i] != 0
        prev = pd.Series(F.bps_to_pips(Gp[i][d], Pp[i][d], PAIR_OF[i]) - F.bps_to_pips(F.commission_bps_per_lot(PAIR_OF[i], COMMISSION_PER_LOT), Pp[i][d], PAIR_OF[i]), index=Gp.index[d])
        yr[i] = pd.concat([prev, net_pips[i]]).groupby(lambda x: x.year).mean()
    yearly = pd.DataFrame(yr)
    display(yearly.round(2)); yearly.to_csv(OUT / "05_yearly.csv")

## 5. A $6,000 prop account through 2026 (fixed size; 5% daily / 10% total limits)

In [ ]:
sets = {"portfolio (all four)": IDS, "HOLDS only": [i for i in IDS if per.loc[i, "verdict"] == "HOLDS"]}
comm_v = pd.Series({i: F.commission_bps_per_lot(PAIR_OF[i], COMMISSION_PER_LOT) for i in IDS})
prop = []
for name, ids in sets.items():
    if not ids:
        continue
    for L in (2, 5):
        o, d, e = F.run_prop_path(G[ids].to_numpy(), M[ids].to_numpy(), comm_v[ids].to_numpy(), np.full(len(ids), float(L)), E0,
                                  F.PropRules(0.05, 0.10, None, 0), len(G), compound=False)
        prop.append({"set": name, "L_each": L, "outcome": o, "day": d if o != "open" else None, "end_equity": round(e, 0)})
display(pd.DataFrame(prop))

In [ ]:
summary = {"study": "FXS06", "synthetic": SYNTHETIC, "window": [str(START), str(END)], "structures": IDS,
           "per_structure": per.reset_index().to_dict(orient="records"), "portfolio": pv, "prop_6k": prop,
           "note": "USDJPY 2026 Tokyo mornings partly seen in GTF 02 (disclosed in discussion 004)."}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
print(json.dumps({"verdicts": per["verdict"].to_dict(), "portfolio": pv["verdict"]}, indent=2))